<a href="https://colab.research.google.com/github/Krishishah7/nlp-learning-series/blob/main/06_llm_and_fine_tuning/21_rag_evaluation/rag_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U sentence-transformers scikit-learn --quiet

In [ ]:
test_data = [
    {
        "question": "What is the capital of France?",
        "ground_truth": "Paris"
    },
    {
        "question": "What is the capital of Germany?",
        "ground_truth": "Berlin"
    },
    {
        "question": "Where is France located?",
        "ground_truth": "Europe"
    }
]

In [ ]:
# -------------------------------
# RAG PIPELINE (SELF-CONTAINED)
# -------------------------------

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Load models
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Knowledge base
documents = [
    "Paris is the capital of France.",
    "France is located in Europe.",
    "Berlin is the capital of Germany.",
    "Python is a programming language."
]

# TF-IDF setup
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(documents)

# Embeddings
embeddings = embed_model.encode(documents)

# -------------------------------
# RAG FUNCTION
# -------------------------------

def rag_pipeline(user_query):

    # Step 1: Rewrite query (stable)
    rewrite_prompt = f"""
Convert the following into a clear question.

Query: {user_query}

Only return the question.
"""
    inputs = tokenizer(rewrite_prompt, return_tensors="pt")
    outputs = model.generate(
        **inputs,
        max_new_tokens=30,
        do_sample=False
    )

    query = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Step 2: Hybrid Retrieval
    query_embedding = embed_model.encode([query])
    query_tfidf = vectorizer.transform([query])

    semantic_scores = cosine_similarity(query_embedding, embeddings)[0]
    keyword_scores = cosine_similarity(query_tfidf, tfidf_matrix)[0]

    hybrid_scores = 0.5 * semantic_scores + 0.5 * keyword_scores

    best_index = hybrid_scores.argmax()
    context = documents[best_index]

    # Step 3: Answer Generation (STRICT)
    prompt = f"""
You are a question answering system.

Use ONLY the context below to answer.

Context: {context}

Question: {query}

Answer in one short sentence.
Do not repeat the question.
"""

    inputs = tokenizer(prompt, return_tensors="pt")

    outputs = model.generate(
        **inputs,
        max_new_tokens=30,
        do_sample=False
    )

    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return answer

In [ ]:
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

def evaluate_rag(test_data):
    scores = []

    for item in test_data:
        question = item["question"]
        ground_truth = item["ground_truth"]

        prediction = rag_pipeline(question)

        emb1 = embed_model.encode([prediction])
        emb2 = embed_model.encode([ground_truth])

        similarity = cosine_similarity(emb1, emb2)[0][0]

        scores.append(similarity)

        print(f"Q: {question}")
        print(f"Prediction: {prediction}")
        print(f"Ground Truth: {ground_truth}")
        print(f"Similarity Score: {similarity:.2f}")
        print("-" * 40)

    avg_score = sum(scores) / len(scores)
    print(f"\nAverage Score: {avg_score:.2f}")

In [8]:
evaluate_rag(test_data)

Q: What is the capital of France?
Prediction: Paris
Ground Truth: Paris
Similarity Score: 1.00
----------------------------------------
Q: What is the capital of Germany?
Prediction: Berlin
Ground Truth: Berlin
Similarity Score: 1.00
----------------------------------------
Q: Where is France located?
Prediction: Europe
Ground Truth: Europe
Similarity Score: 1.00
----------------------------------------

Average Score: 1.00
